# **On-device Image Classifier**

Investigate how an image-classification model can be designed for direct inference on resource-constrained embedded hardware while maintaining useful classification accuracy.

## Initial Task

Classify images into:

1. Apple
2. Banana
3. Orange

## Development Environment

* Google Colab

## Target Hardware

**AI-Thinker ESP32-CAM**

| Resource       |                       Specification |
| -------------- | ----------------------------------: |
| CPU            | Dual-core Xtensa LX6, up to 240 MHz |
| Flash          |                                4 MB |
| Internal SRAM  |                              520 KB |
| External PSRAM |                                4 MB |
| Wi-Fi          |                        802.11 b/g/n |
| Bluetooth      |                       Bluetooth 4.2 |

## Project Objective

**Prepare an image-classification model for direct inference on the target embedded device while maintaining useful classification accuracy.**

## Design Constraints

The experiment will investigate the trade-offs between:

* Classification accuracy
* Model size
* Parameter count
* Activation memory
* Flash requirements
* RAM requirements

# Set the Environmental conditions

In [1]:
import os
import random
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

print("TensorFlow version:", tf.__version__)

TensorFlow version: 2.21.0
TensorFlow version: 2.21.0


# Read & Understand the Dataset

## *Upload the Dataset*

In [2]:
import zipfile
from pathlib import Path

zip_path = Path("/content/Fruits_Dataset.zip")
dataset_path = Path("/content/Fruits_Dataset")
with zipfile.ZipFile(zip_path) as z:
  z.extractall("/content")
os.remove(zip_path)
print("Dataset path:", dataset_path)

Dataset path: /content/Fruits_Dataset


## *Verify the file*

In [3]:
from PIL import Image

classes = ["Apple", "Banana", "Orange"]
for split in ["Train", "Validation", "Test"]:
  print(f"\n{split}")
  for class_name in classes:
    folder = Path(dataset_path) / split / class_name
    count = len(list(folder.glob("*.jpg")))
    imagePath = list(folder.glob("*.jpg"))[0]
    image = Image.open(imagePath)
    print(f"{class_name}: {count}, Image Size: {image.size}, Image Mode: {image.mode}")


Train
Apple: 350, Image Size: (100, 100), Image Mode: RGB
Banana: 350, Image Size: (100, 100), Image Mode: RGB
Orange: 350, Image Size: (100, 100), Image Mode: RGB

Validation
Apple: 75, Image Size: (100, 100), Image Mode: RGB
Banana: 75, Image Size: (100, 100), Image Mode: RGB
Orange: 75, Image Size: (100, 100), Image Mode: RGB

Test
Apple: 75, Image Size: (100, 100), Image Mode: RGB
Banana: 75, Image Size: (100, 100), Image Mode: RGB
Orange: 75, Image Size: (100, 100), Image Mode: RGB


# Modelling

## Create the Model

In [4]:
# design CNN model

SEED = 42
MAX_EPOCHS = 100
BATCH_SIZE = 32
IMG_SIZE = (100, 100)

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

model = keras.Sequential([
  layers.Input(shape=(100, 100, 3)),
  layers.Rescaling(1./255),
  layers.Conv2D(16, (3, 3), padding="same", activation="relu"),
  layers.MaxPooling2D((2, 2)),
  layers.Conv2D(32, (3, 3), padding="same", activation="relu"),
  layers.MaxPooling2D((2, 2)),
  layers.Flatten(),
  layers.Dense(64, activation="relu"),
  layers.Dense(3, activation="softmax")
])

In [5]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ rescaling (Rescaling)           │ (None, 100, 100, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 100, 100, 16)   │           448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 50, 50, 16)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 50, 50, 32)     │         4,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 25, 25, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 20000)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │     1,280,064 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 3)              │           195 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,285,347 (4.90 MB)

 Trainable params: 1,285,347 (4.90 MB)

 Non-trainable params: 0 (0.00 B)

In [6]:
model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss=keras.losses.SparseCategoricalCrossentropy(), metrics=["accuracy"])

In [7]:
early_stopping = keras.callbacks.EarlyStopping(monitor="val_loss",
    patience=5, restore_best_weights=True)

In [8]:
def load_image(image_path, label):
  image = tf.io.read_file(image_path)
  image = tf.image.decode_jpeg(image, channels=3)
  image = tf.image.resize(image, [100, 100])
  return image, label

In [9]:
train_ds = tf.keras.utils.image_dataset_from_directory("/content/Fruits_Dataset/Train",
    image_size=IMG_SIZE, batch_size=BATCH_SIZE, shuffle=True, seed=SEED)

val_ds = tf.keras.utils.image_dataset_from_directory("/content/Fruits_Dataset/Validation",
    image_size=IMG_SIZE, batch_size=BATCH_SIZE, shuffle=False)

test_ds = tf.keras.utils.image_dataset_from_directory("/content/Fruits_Dataset/Test",
    image_size=IMG_SIZE, batch_size=BATCH_SIZE, shuffle=False)

Found 1050 files belonging to 3 classes.
Found 225 files belonging to 3 classes.
Found 225 files belonging to 3 classes.


In [10]:
history = model.fit(train_ds, validation_data=val_ds, epochs=MAX_EPOCHS,
                                                     callbacks=[early_stopping])

Epoch 1/100
33/33 ━━━━━━━━━━━━━━━━━━━━ 11s 280ms/step - accuracy: 0.7743 - loss: 0.5212 - val_accuracy: 0.9289 - val_loss: 0.2380
Epoch 2/100
33/33 ━━━━━━━━━━━━━━━━━━━━ 9s 285ms/step - accuracy: 0.9610 - loss: 0.1390 - val_accuracy: 0.9689 - val_loss: 0.1151
Epoch 3/100
33/33 ━━━━━━━━━━━━━━━━━━━━ 10s 285ms/step - accuracy: 0.9867 - loss: 0.0544 - val_accuracy: 0.9733 - val_loss: 0.0641
Epoch 4/100
33/33 ━━━━━━━━━━━━━━━━━━━━ 9s 240ms/step - accuracy: 0.9933 - loss: 0.0286 - val_accuracy: 0.9822 - val_loss: 0.0715
Epoch 5/100
33/33 ━━━━━━━━━━━━━━━━━━━━ 9s 288ms/step - accuracy: 0.9962 - loss: 0.0137 - val_accuracy: 0.9822 - val_loss: 0.0496
Epoch 6/100
33/33 ━━━━━━━━━━━━━━━━━━━━ 10s 285ms/step - accuracy: 0.9990 - loss: 0.0067 - val_accuracy: 0.9778 - val_loss: 0.0794
Epoch 7/100
33/33 ━━━━━━━━━━━━━━━━━━━━ 8s 245ms/step - accuracy: 1.0000 - loss: 0.0028 - val_accuracy: 0.9867 - val_loss: 0.0589
Epoch 8/100
33/33 ━━━━━━━━━━━━━━━━━━━━ 10s 245ms/step - accuracy: 1.0000 - loss: 0.0013 - val_

## Convert TFlite to INT 8

In [11]:
def representative_dataset():
  for images, _ in train_ds.take(100):
    for image in images:
      image = tf.expand_dims(image, axis=0)
      yield [tf.cast(image, tf.float32)]

In [12]:
converter_int8 = tf.lite.TFLiteConverter.from_keras_model(model)
converter_int8.optimizations = [tf.lite.Optimize.DEFAULT]
converter_int8.representative_dataset = representative_dataset
converter_int8.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]

converter_int8.inference_input_type = tf.int8
converter_int8.inference_output_type = tf.int8

int8_tflite_model = converter_int8.convert()
int8_path = "/content/baseline_int8.tflite"
with open(int8_path, "wb") as f:
  f.write(int8_tflite_model)
print("INT8 model saved:", int8_path)

Saved artifact at '/tmp/tmp_6a1zklu'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 100, 100, 3), dtype=tf.float32, name='keras_tensor')
Output Type:
  TensorSpec(shape=(None, 3), dtype=tf.float32, name=None)
Captures:
  137163229088464: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137163229089424: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137163229089040: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137163229089808: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137163229089616: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137163229089232: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137163229090000: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137163229090576: TensorSpec(shape=(), dtype=tf.resource, name=None)


/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/convert.py:846: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(


INT8 model saved: /content/baseline_int8.tflite


In [13]:
size_bytes = os.path.getsize(int8_path)
size_kb = size_bytes / 1024
size_mb = size_bytes / (1024 * 1024)

print(f"INT8 model size: {size_bytes:,} bytes")
print(f"INT8 model size: {size_kb:.2f} KB")
print(f"INT8 model size: {size_mb:.2f} MB")

INT8 model size: 1,293,432 bytes
INT8 model size: 1263.12 KB
INT8 model size: 1.23 MB


In [14]:
interpreter_int8 = tf.lite.Interpreter(model_path=int8_path)
interpreter_int8.allocate_tensors()

input_details_int8 = interpreter_int8.get_input_details()
output_details_int8 = interpreter_int8.get_output_details()

print("INT8 TFLite Model")
print("-" * 50)

print("Input:")
print("  Shape :", input_details_int8[0]["shape"])
print("  Dtype :", input_details_int8[0]["dtype"])
print("  Scale :", input_details_int8[0]["quantization"][0])
print("  Zero Point :", input_details_int8[0]["quantization"][1])

print("\nOutput:")
print("  Shape :", output_details_int8[0]["shape"])
print("  Dtype :", output_details_int8[0]["dtype"])
print("  Scale :", output_details_int8[0]["quantization"][0])
print("  Zero Point :", output_details_int8[0]["quantization"][1])

INT8 TFLite Model
--------------------------------------------------
Input:
  Shape : [  1 100 100   3]
  Dtype : <class 'numpy.int8'>
  Scale : 1.0
  Zero Point : -128

Output:
  Shape : [1 3]
  Dtype : <class 'numpy.int8'>
  Scale : 0.00390625
  Zero Point : -128


/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


## Evaluate the Model

In [15]:
y_true_int8 = []
y_pred_int8 = []
input_scale, input_zero_point = input_details_int8[0]["quantization"]
for images, labels in test_ds:
  images_np = images.numpy()
  labels_np = labels.numpy()
  for i in range(len(images_np)):
    image = images_np[i]
    image_int8 = np.round(image / input_scale + input_zero_point).astype(np.int8)
    input_data = np.expand_dims(image_int8, axis=0)
    interpreter_int8.set_tensor(input_details_int8[0]["index"], input_data)
    interpreter_int8.invoke()
    output = interpreter_int8.get_tensor(output_details_int8[0]["index"])
    predicted_class = np.argmax(output[0])
    y_true_int8.append(labels_np[i])
    y_pred_int8.append(predicted_class)

y_true_int8 = np.array(y_true_int8)
y_pred_int8 = np.array(y_pred_int8)
accuracy_int8 = accuracy_score(y_true_int8, y_pred_int8)
print(f"INT8 Test Accuracy: {accuracy_int8 * 100:.2f}%")
print("\nConfusion Matrix:")
print(confusion_matrix(y_true_int8, y_pred_int8))
print("\nClassification Report:")
print(classification_report(y_true_int8, y_pred_int8, target_names=test_ds.class_names))

INT8 Test Accuracy: 99.11%

Confusion Matrix:
[[73  1  1]
 [ 0 75  0]
 [ 0  0 75]]

Classification Report:
              precision    recall  f1-score   support

       Apple       1.00      0.97      0.99        75
      Banana       0.99      1.00      0.99        75
      Orange       0.99      1.00      0.99        75

    accuracy                           0.99       225
   macro avg       0.99      0.99      0.99       225
weighted avg       0.99      0.99      0.99       225



# Observations

The model has been converted to **full INT8 quantization** for deployment-oriented inference on the target embedded device.

| Metric | INT8 Model |
|---|---:|
| Parameters | 1,285,347 |
| Weight representation | INT8 |
| Activation representation | INT8 |
| Test accuracy | 99.11% |
| Test errors | 2 / 225 |
| TFLite model size | 1.23 MiB |
| Largest activation | ~156 KiB |

### What changed?

The model is now represented using **INT8 weights and INT8 activations**, with INT32 used where required for intermediate computations such as accumulators.

The resulting TFLite model is approximately **1.23 MiB** in size, while the largest activation tensor requires approximately **156 KiB** when represented as INT8.

### What about accuracy?

The INT8 model achieves **99.11% test accuracy** on the fixed test set.

The same **2 out of 225** test images are misclassified.

For this dataset, calibration, model, and test set, the INT8 conversion therefore introduces **no observed accuracy degradation**.

### What does this mean?

The INT8 model is now a deployment candidate for the target embedded device.

At the model/tensor level:

- **Flash:** ~1.23 MiB serialized model
- **Largest activation:** ~156 KiB
- **Parameters:** 1,285,347
- **Inference representation:** INT8

These measurements indicate that the model is designed within the nominal Flash and SRAM constraints of the target hardware.

> **INT8 conclusion:** Full integer quantization provides a compact model representation suitable for investigating direct on-device inference while maintaining the same observed classification accuracy on the fixed test set.